# FDKD — 4-Model Pipeline Demo
## Google Colab: Miniforge + Conda + Export + Server

Pipeline: **Teacher** (Swin-B) → **DKD** (direct distill) → **TAKD** (Swin→R152→R18) → **Baseline** (fully trained)

Chạy từng cell từ trên xuống dưới.

In [ ]:
# ── Cell 1: Cài Miniforge + tạo môi trường Conda Python 3.10 ──
!wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
!bash Miniforge3-Linux-x86_64.sh -b -p /opt/conda 2>/dev/null
import os
os.environ['PATH'] = '/opt/conda/bin:' + os.environ['PATH']

!conda remove -n openmmlab --all -y -q 2>/dev/null
!conda create --name openmmlab python=3.10 -y -q

print("✅ Miniforge + conda env 'openmmlab' ready")

In [ ]:
# ── Cell 2: Clone repo + mount Drive ──
%cd /content
import os
if not os.path.exists('/content/FDKD-Fusion-Decoupled-Knowledge-Distillation'):
    !git clone https://github.com/MingNhayMua/FDKD-Fusion-Decoupled-Knowledge-Distillation.git
%cd /content/FDKD-Fusion-Decoupled-Knowledge-Distillation

from google.colab import drive
drive.mount('/content/drive')

CKPT_DIR = "/content/drive/MyDrive/CS338-checkpoint"
print(f"Checkpoint dir: {CKPT_DIR}")
print(f"Exists: {os.path.isdir(CKPT_DIR)}")
if os.path.isdir(CKPT_DIR):
    !ls "$CKPT_DIR"

In [ ]:
# ── Cell 3: Cài PyTorch + OpenMMLab + backend deps ──
!conda run -n openmmlab pip install torch==2.1.0 torchvision==0.16.0 "numpy<2" --index-url https://download.pytorch.org/whl/cu121 -q
!conda run -n openmmlab pip install mmcv==2.1.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html "numpy<2" -q
!conda run -n openmmlab pip install -U openmim "numpy<2" -q
!conda run -n openmmlab mim install "mmpretrain>=1.0.0" -q
!conda run -n openmmlab pip install -r backend/requirements.txt "numpy<2" -q

print("✅ All dependencies installed")

In [ ]:
# ── Cell 4: Debug key mapping của DKD, TAKD, Baseline checkpoints ──
!conda run --no-capture-output -n openmmlab python -c "
import torch, torch.nn as nn
from torchvision import models as tv_models

CKPT = '$CKPT_DIR'

for name in ['swinb_r18_clean.pth', 'disilledr152_r18_clean.pth', 'r18_fully.pth']:
    path = f'{CKPT}/{name}'
    if not __import__('os').path.exists(path):
        print(f'=== {name}: NOT FOUND ===')
        continue
    print(f'=== {name} ===')
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    sd = ckpt.get('state_dict', ckpt)
    sample = list(sd.keys())[:5]
    print(f'  First 5 keys: {sample}')
    
    model = tv_models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 200)
    model_keys = set(model.state_dict().keys())
    
    mapped = {}
    for k, v in sd.items():
        new = k
        if new.startswith('backbone.'): new = new[len('backbone.'):]
        elif new.startswith('head.'): new = new[len('head.'):]
        mapped[new] = v
    
    mapped_keys = set(mapped.keys())
    matched = model_keys & mapped_keys
    missing = model_keys - mapped_keys
    unexpected = mapped_keys - model_keys
    print(f'  Matched: {len(matched)}, Missing: {len(missing)}, Unexpected: {len(unexpected)}')
    if missing: print(f'  Missing (sample): {list(missing)[:5]}')
    if unexpected: print(f'  Unexpected: {list(unexpected)[:5]}')
    
    model.load_state_dict(mapped, strict=False)
    model.eval()
    with torch.no_grad():
        out = model(torch.randn(1, 3, 224, 224))
        probs = torch.softmax(out, dim=-1)
        top3 = probs.topk(3)
        print(f'  Top-3: {[(i.item(), round(v.item(),4)) for i, v in zip(top3.indices[0], top3.values[0])]}')
    print()
"
print("✅ Key mapping check done")

In [ ]:
# ── Cell 5: Export 4 models to TorchScript ──
!conda run --no-capture-output -n openmmlab python export_models.py --checkpoint-dir "$CKPT_DIR"

In [ ]:
# ── Cell 6: Start server ──
NGROK_TOKEN = ""  # ← điền ngrok auth token

!conda run --no-capture-output -n openmmlab python run_colab.py --checkpoint-dir "$CKPT_DIR" --token "$NGROK_TOKEN" --port 8000

## Sau khi chạy xong

- Copy **public URL** (dạng `https://xxxx.ngrok-free.app`) từ output Cell 6
- Paste vào frontend connection input

## Lưu ý
- Cell 4 debug key mapping trước khi export — verify không thiếu key
- Cell 5 có thể skip nếu `*_traced.pt` đã tồn tại
- Điền `NGROK_TOKEN` ở Cell 6 để tránh giới hạn 2h free tier